# Orchestrated ML Pipeline with Mage: Iris Classification
### Data Engineering Assignment: Part C Implementation
This notebook executes and validates the modular machine learning pipeline:
- **Architecture**: `load_iris` -> `data_validation` -> `prepare_data` -> `train_model` -> (`save_results` & `evaluate_model`)
- **Key Concept**: Test-driven data orchestration where block-level `@test` quality assertions act as hard gating mechanisms.


## 1. Setup & Block Definitions


In [3]:
import os
import pandas as pd
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Imports ready.")


Imports ready.


## 2. Part C1: Pipeline Block Graph
```mermaid
graph TD
    A[load_iris<br/>data_loader] -->|@test len == 150| B[data_validation<br/>transformer]
    B -->|@test no nulls & 3 classes| C[prepare_data<br/>transformer]
    C -->|@test split column exists| D[train_model<br/>transformer]
    D -->|@test accuracy >= 0.80| E[save_results<br/>data_exporter]
    D -->|@test accuracy >= 0.80| F[evaluate_model<br/>transformer]
```


In [5]:
# Simulating Mage Pipeline Execution: mage run iris_project iris_classifier --test
def run_pipeline(max_depth=3):
    print(f"=== RUNNING PIPELINE: iris_classifier (max_depth={max_depth}) ===")
    # Block 1: load_iris
    iris = load_iris(as_frame=True)
    df = iris.frame
    assert len(df) == 150, "Failed load_iris test"
    print("✓ Block 1 [load_iris]: Loaded 150 rows. Test passed.")

    # Block 2 (Custom C4): data_validation
    assert df.isnull().sum().sum() == 0 and df['target'].nunique() == 3
    print("✓ Block 2 [data_validation]: 0 missing values, 3 distinct classes. Test passed.")

    # Block 3: prepare_data
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    train_df = train_df.copy(); test_df = test_df.copy()
    train_df['split'] = 'train'; test_df['split'] = 'test'
    full_df = pd.concat([train_df, test_df])
    assert 'split' in full_df.columns
    print(f"✓ Block 3 [prepare_data]: 120 train / 30 test. Test passed.")

    # Block 4: train_model
    features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model.fit(train_df[features], train_df['target'])
    test_df['predicted'] = model.predict(test_df[features])
    acc = (test_df['predicted'] == test_df['target']).mean()
    print(f"  Block 4 [train_model]: max_depth={max_depth} -> Test accuracy={acc:.4f}")

    # Quality Gate
    if acc < 0.80:
        print(f"✗ Block 4 [train_model] @test FAILED: Accuracy {acc:.4f} < 0.80 threshold!")
        print("  [PIPELINE ABORTED] Downstream blocks (save_results, evaluate_model) will NOT execute.")
        return False, acc

    print("✓ Block 4 [train_model] @test PASSED: Accuracy meets threshold.")

    # Block 5: save_results
    test_df.to_csv("iris_predictions.csv", index=False)
    print("✓ Block 5 [save_results]: Exported 30 predictions to iris_predictions.csv")

    # Block 6: evaluate_model
    macro_f1 = f1_score(test_df['target'], test_df['predicted'], average='macro')
    print(f"✓ Block 6 [evaluate_model]: Evaluated metrics (Macro F1 = {macro_f1:.4f})")
    return True, acc

run_pipeline(max_depth=3)


=== RUNNING PIPELINE: iris_classifier (max_depth=3) ===
✓ Block 1 [load_iris]: Loaded 150 rows. Test passed.
✓ Block 2 [data_validation]: 0 missing values, 3 distinct classes. Test passed.
✓ Block 3 [prepare_data]: 120 train / 30 test. Test passed.
  Block 4 [train_model]: max_depth=3 -> Test accuracy=1.0000
✓ Block 4 [train_model] @test PASSED: Accuracy meets threshold.
✓ Block 5 [save_results]: Exported 30 predictions to iris_predictions.csv
✓ Block 6 [evaluate_model]: Evaluated metrics (Macro F1 = 1.0000)


## 3. Part C2: Tests as a Gate Experiment
Running the pipeline across `max_depth = 1, 2, 3, 4, 5`.
Notice that at `max_depth = 1`, the accuracy is **0.6333**, failing the `@test` threshold of `0.80`.
Because the gate fails, Mage aborts downstream execution, preventing `iris_predictions.csv` from being written.


In [7]:
print("max_depth | Test accuracy | test_accuracy passed? | iris_predictions.csv created?")
print("----------+---------------+-----------------------+-----------------------------")
for d in [1, 2, 3, 4, 5]:
    if os.path.exists("iris_predictions.csv"):
        os.remove("iris_predictions.csv")
    passed, acc = run_pipeline(max_depth=d)
    created = os.path.exists("iris_predictions.csv")
    print(f"{d:9d} | {acc:13.4f} | {'Yes' if passed else 'No':21s} | {'Yes' if created else 'No'}")


max_depth | Test accuracy | test_accuracy passed? | iris_predictions.csv created?
----------+---------------+-----------------------+-----------------------------
        1 |        0.6333 | No                    | No
        2 |        0.9667 | Yes                   | Yes
        3 |        1.0000 | Yes                   | Yes
        4 |        1.0000 | Yes                   | Yes
        5 |        1.0000 | Yes                   | Yes


## 4. Part C3: Evaluation Branch
Confusion Matrix and Macro Classification Metrics generated by `evaluate_model`:


In [9]:
from iris_project.transformers.evaluate_model import evaluate

iris = load_iris(as_frame=True)
train_df, test_df = train_test_split(iris.frame, test_size=0.2, random_state=42)
features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(train_df[features], train_df['target'])
test_df = test_df.copy()
test_df['predicted'] = model.predict(test_df[features])

metrics = evaluate(test_df)


Confusion Matrix:
Predicted  0  1  2
Actual            
0         10  0  0
1          0  9  0
2          0  0 11

Model Evaluation Metrics:
      metric  value
0   accuracy    1.0
1  precision    1.0
2     recall    1.0
3   f1_score    1.0


## 5. Part C4: Custom Block (`data_validation`)
A dataset validator positioned downstream of `load_iris` verifying null-freedom and 3 target classes.


In [11]:
from iris_project.transformers.data_validation import validate_dataset, test_data_integrity

df = load_iris(as_frame=True).frame
out = validate_dataset(df)
test_data_integrity(out)
print("✓ Custom block test_data_integrity passed successfully.")


Data Validation: 0 missing values found across 150 rows.
Target distribution: 3 distinct classes present.
✓ Custom block test_data_integrity passed successfully.
